In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("../data/AmesHousing.csv")

print(df.shape)

df.head()

(2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
selected_features = [

    "Overall Qual",
    "Overall Cond",
    "Year Built",
    "Year Remod/Add",

    "Lot Area",
    "Lot Frontage",
    "Gr Liv Area",
    "Total Bsmt SF",
    "Garage Area",
    "Garage Cars",

    "Full Bath",
    "Half Bath",
    "Bedroom AbvGr",
    "TotRms AbvGrd",
    "Fireplaces",

    "Kitchen Qual",
    "Exter Qual",
    "Heating QC",

    "Neighborhood",
    "House Style",
    "Bldg Type",
    "Foundation",

    "Bsmt Qual",
    "Garage Finish",
    "Central Air"
]

target = "SalePrice"

df = df[selected_features + [target]]

print(df.shape)
df.head()

(2930, 26)


,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Lot Area,Lot Frontage,Gr Liv Area,Total Bsmt SF,Garage Area,Garage Cars,...,Exter Qual,Heating QC,Neighborhood,House Style,Bldg Type,Foundation,Bsmt Qual,Garage Finish,Central Air,SalePrice
0,6,5,1960,1960,31770,141.0,1656,1080.0,528.0,2.0,...,TA,Fa,NAmes,1Story,1Fam,CBlock,TA,Fin,Y,215000
1,5,6,1961,1961,11622,80.0,896,882.0,730.0,1.0,...,TA,TA,NAmes,1Story,1Fam,CBlock,TA,Unf,Y,105000
2,6,6,1958,1958,14267,81.0,1329,1329.0,312.0,1.0,...,TA,TA,NAmes,1Story,1Fam,CBlock,TA,Unf,Y,172000
3,7,5,1968,1968,11160,93.0,2110,2110.0,522.0,2.0,...,Gd,Ex,NAmes,1Story,1Fam,CBlock,TA,Fin,Y,244000
4,5,5,1997,1998,13830,74.0,1629,928.0,482.0,2.0,...,TA,Gd,Gilbert,2Story,1Fam,PConc,Gd,Fin,Y,189900


In [4]:
# ==========================================
# Handle Missing Values
# ==========================================

# Numerical Columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols.remove("SalePrice")

# Categorical Columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Fill numerical missing values with median
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with "None"
for col in categorical_cols:
    df[col] = df[col].fillna("None")

print("Remaining Missing Values:", df.isnull().sum().sum())

Remaining Missing Values: 0


In [5]:
# ==========================================
# Feature Engineering
# ==========================================

df["Total Bathrooms"] = (
    df["Full Bath"] +
    0.5 * df["Half Bath"]
)

df["Total Living Area"] = (
    df["Gr Liv Area"] +
    df["Total Bsmt SF"]
)

df.head()

,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Lot Area,Lot Frontage,Gr Liv Area,Total Bsmt SF,Garage Area,Garage Cars,...,Neighborhood,House Style,Bldg Type,Foundation,Bsmt Qual,Garage Finish,Central Air,SalePrice,Total Bathrooms,Total Living Area
0,6,5,1960,1960,31770,141.0,1656,1080.0,528.0,2.0,...,NAmes,1Story,1Fam,CBlock,TA,Fin,Y,215000,1.0,2736.0
1,5,6,1961,1961,11622,80.0,896,882.0,730.0,1.0,...,NAmes,1Story,1Fam,CBlock,TA,Unf,Y,105000,1.0,1778.0
2,6,6,1958,1958,14267,81.0,1329,1329.0,312.0,1.0,...,NAmes,1Story,1Fam,CBlock,TA,Unf,Y,172000,1.5,2658.0
3,7,5,1968,1968,11160,93.0,2110,2110.0,522.0,2.0,...,NAmes,1Story,1Fam,CBlock,TA,Fin,Y,244000,2.5,4220.0
4,5,5,1997,1998,13830,74.0,1629,928.0,482.0,2.0,...,Gilbert,2Story,1Fam,PConc,Gd,Fin,Y,189900,2.5,2557.0


In [6]:
# ==========================================
# One-Hot Encoding
# ==========================================

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X = pd.get_dummies(X)

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

X.head()

Feature Matrix Shape: (2930, 90)
Target Shape: (2930,)


,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Lot Area,Lot Frontage,Gr Liv Area,Total Bsmt SF,Garage Area,Garage Cars,...,Bsmt Qual_Gd,Bsmt Qual_None,Bsmt Qual_Po,Bsmt Qual_TA,Garage Finish_Fin,Garage Finish_None,Garage Finish_RFn,Garage Finish_Unf,Central Air_N,Central Air_Y
0,6,5,1960,1960,31770,141.0,1656,1080.0,528.0,2.0,...,False,False,False,True,True,False,False,False,False,True
1,5,6,1961,1961,11622,80.0,896,882.0,730.0,1.0,...,False,False,False,True,False,False,False,True,False,True
2,6,6,1958,1958,14267,81.0,1329,1329.0,312.0,1.0,...,False,False,False,True,False,False,False,True,False,True
3,7,5,1968,1968,11160,93.0,2110,2110.0,522.0,2.0,...,False,False,False,True,True,False,False,False,False,True
4,5,5,1997,1998,13830,74.0,1629,928.0,482.0,2.0,...,True,False,False,False,True,False,False,False,False,True


In [7]:
# ==========================================
# Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (2344, 90)
Testing Shape  : (586, 90)


In [8]:
# ==========================================
# Feature Scaling
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed successfully!")

Scaling completed successfully!


In [9]:
# ==========================================
# Train Models
# ==========================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
}

results = []

best_model = None
best_r2 = -999
best_model_name = ""

for name, model in models.items():

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    predictions = model.predict(X_test_scaled)

    # Metrics
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "R2 Score": round(r2,4)
    })

    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_model_name = name

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="R2 Score",
    ascending=False,
    inplace=True
)

results_df

c:\Users\docto\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.676e+11, tolerance: 1.394e+09
  model = cd_fast.enet_coordinate_descent(


,Model,MAE,RMSE,R2 Score
4,Random Forest,15886.92,27765.94,0.9038
1,Ridge,17489.96,30435.54,0.8845
0,Linear Regression,17493.11,30439.61,0.8844
2,Lasso,17498.14,30441.10,0.8844
3,Decision Tree,24687.81,42610.13,0.7735


In [10]:
print("="*50)
print("Best Model :", best_model_name)
print("Best R² :", round(best_r2,4))
print("="*50)

Best Model : Random Forest
Best R² : 0.9038


In [11]:
# ==========================================
# Save Final Files
# ==========================================

joblib.dump(best_model, "../models/best_model.pkl")

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

joblib.dump(
    X.columns.tolist(),
    "../models/feature_columns.pkl"
)

print("✅ best_model.pkl saved")
print("✅ scaler.pkl saved")
print("✅ feature_columns.pkl saved")

✅ best_model.pkl saved
✅ scaler.pkl saved
✅ feature_columns.pkl saved
